# Model A — independent YOLO26 six-class baseline

**Knowledge-Preserving Vocabulary Expansion for Frame-Based BISINDO Classification: A Comparative Study of Independent and Sequential YOLO26 Training**

## 1. Research objective and Model A definition
Independently fine-tune official ImageNet-pretrained `yolo26n-cls.pt` on saya, kamu, senang, bingung, marah, apa_kabar. One full-context RGB frame per source video; no temporal translation or Model B/C.

**Revised researcher instruction, 2026-11-10:** complete semantic approval is not mandatory for this exploratory baseline. Technical integrity, traceability, all 1,800 videos and the unchanged 1,440/360 split are mandatory. Unverified/flagged frames retain their video-derived labels and quality concerns. Automatic selection does not establish semantic validity.

See `reports/automatic_selection_revision/REPORT.md` for the current audit, execution and actual results. Historical construction reports are preserved in `reports/automatic_selection_revision/before/`.

## 2. Environment inspection
Use the existing `.venv-bisindo` kernel. The system Python lacks Ultralytics. This workflow checks the audited Ultralytics 8.4.197 API; re-audit before changing that version.

In [ ]:
from pathlib import Path
import sys, json
# Works from repository root or this notebook's folder; no fixed machine path.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/model_a/model_a.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open from repository root or experiments/model_a')
WORK = ROOT / 'experiments/model_a'
sys.path.insert(0, str(WORK))
import model_a as a
from IPython.display import display
import pandas as pd
ENVIRONMENT = a.environment()
a.write_json(WORK / 'reports/environment.json', ENVIRONMENT)
display(ENVIRONMENT)

## 3. Configuration and random seeds
Execution flags default to **False** to prevent accidental repeated fitting. This task authorizes one official GPU fit after technical QA. Set one flag True only to launch a new separate run. The notebook otherwise prepares/audits the dataset and displays an existing completed official run, if available. Settings remain initial decisions, not tuned optima. Deterministic algorithms use warning mode where unavailable; cross-hardware bitwise equality is not guaranteed.

In [ ]:
RUN_TRAINING = False  # Explicit opt-in: full official Model A; requires full technical QA.
RUN_SMOKE_TEST = False  # Explicit opt-in: 1 epoch, 2 existing frames/class/split; technical only.
LOAD_PRETRAINED = True  # Downloads and verifies official asset; never starts training.
assert not (RUN_TRAINING and RUN_SMOKE_TEST), 'Choose one execution mode'
CONFIG = json.loads((WORK / 'config/model_a.json').read_text())
a.seeds(CONFIG['training']['seed'])
IMG_SIZE = CONFIG['training']['imgsz']  # Initial 224; edit config for a separately recorded experiment.
display(CONFIG)

## 4. Dataset discovery and source manifests
Raw internal videos: `04_dataset/train/`. Frozen identities/splits: `02_exp/arsip_pipeline/experiment_yolo26_v3/split_lock.csv`. No split is regenerated. Exactly 1,800 six-class source videos.

`frame_selection.py` reuses existing full-frame candidates and audited `frame_candidates.assess_hands` scoring. Policy declared in `config/frame_selection_policy_v1.json`: readable/integrity verified; eligible first; detected hands preferred; fewer technical flags; higher minimum hand-ROI focus; lower frame index; candidate ID. Identical across classes and splits, with no accuracy-dependent selection. A flagged readable fallback is retained if no eligible candidate exists.

All 480 historical selections agree with this policy. The 1,320 gaps came from historical pilot subsampling, not absent candidate frames. Cache files lack extraction timestamps; original record/image mtimes and hashes are preserved as observations, never invented timestamps. No Kaggle images or scores are inspected.

In [ ]:
from frame_selection import build, contact_sheets, MANIFEST
if not MANIFEST.exists():
    contact_sheets(build())
AUDIT, FRAME_TEMPLATE = a.audit(hash_videos=True)
SOURCE = a.read_csv(WORK / 'manifests/source_inventory.csv')
display(pd.read_csv(WORK / 'reports/split_counts.csv'))
display(json.loads((WORK / 'reports/automatic_selection_revision/selection_summary.json').read_text()))

## 5. Frozen-split verification and leakage checks
The audit validates unique IDs, paths, video hashes, canonical labels, source/split identity, physical internal inventory and frozen inventory equality. All 1,800 source files are hashed. Identical source bytes across train/val are rejected; included selected JPEG bytes and decoded RGB are checked again during preparation. This does not prove absence of near-duplicates or overlapping video segments.

Internal validation is **not signer-independent**. Same researcher-signers may occur in both splits. Historical Kaggle rows are read only as frozen identity metadata, never as training/validation samples or preprocessing targets.

In [ ]:
assert len(SOURCE) == 1800 and set(SOURCE.class_name) == set(a.CLASSES)
assert set(SOURCE.source) == {'train'} and set(SOURCE.split) == {'train', 'val'}
assert not set(SOURCE[SOURCE.split.eq('train')].video_sha256) & set(SOURCE[SOURCE.split.eq('val')].video_sha256)
assert a.sha(a.LOCK) == AUDIT['protected_input_hashes'][str(a.LOCK.relative_to(ROOT))]
print('Frozen identity/source checks PASS. Signer/near-duplicate independence UNVERIFIED.')

## 6. Six-class dataset preparation
`manifests/automatic_frame_selection_v1.csv` contains exactly one cached frame for each of the 1,800 source videos. Source identities, labels and frozen splits must match. All selected bytes are hashed and RGB-decoded; cross-split byte/pixel duplicates fail preparation. Dataset copies are immutable/content-addressed. No exclusions or relabeling.

`frame_decisions.csv` is retained as historical provenance; its former universal manual-approval gate is superseded. Semantic concern flags remain visible, including actual historical reviewer decisions. `flagged_video_review.csv` joins all 49 concern videos to the current and pilot candidate IDs, technical status, reviewer labels/reasons and original labels. Readable images may still be semantically ambiguous or wrong-label; no automatic correctness claim.

Contact sheets cover all classes and both splits, stratified by eligible/flagged status. Two additional sheets cover all 49 concern videos. Full manual approval is not required.

In [ ]:
DATASET, PREPARED_ROWS, DATASET_FINGERPRINT = a.prepare(smoke=RUN_SMOKE_TEST)
display(json.loads((DATASET / 'quality_assurance.json').read_text()))
LOADING = a.verify_loading(DATASET)
display(LOADING)
display(pd.read_csv(WORK / 'reports/automatic_selection_revision/flagged_video_review.csv').head(12))
print('Contact sheets:', WORK / 'reports/automatic_selection_revision/contact_sheets')

## 7. Image preprocessing and visual sanity checks
Installed default training uses RandomResizedCrop; default validation uses Resize + CenterCrop. Both can remove gesture regions. The small `ModelATrainer`, `ModelAValidator` and `ModelAPredictor` overrides share `LetterboxRGB` instead.

**Every enabled transform:** convert PIL image to RGB; bilinear aspect-preserving resize; centered constant `(114,114,114)` padding to 224²; float tensor /255; normalize mean=(0,0,0), std=(1,1,1), matching this Ultralytics pretrained pipeline. No random augmentation: no flips, rotation, crop, color jitter, perspective, erasing, MixUp or CutMix. Validation/inference are deterministic. Resizing cannot restore context already missing from the source frame.

[Official Ultralytics classification documentation](https://docs.ultralytics.com/tasks/classify/). The following images are an inspection aid, **not** semantic approval.

In [ ]:
figure = a.sanity_plot(FRAME_TEMPLATE, IMG_SIZE)
display(figure)
import matplotlib.pyplot as plt
plt.close(figure)

## 8. Official pretrained YOLO26 initialization
Fresh asset from Ultralytics/assets v8.4.0. SHA256 pinned and verified before loading; task must be classify with 1,000 ImageNet outputs. Historical checkpoints are never searched. Network failure is a real blocker; no fallback to trained weights.

In [ ]:
PRETRAINED_PATH = None
PRETRAINED_MODEL = None
if LOAD_PRETRAINED:
    PRETRAINED_PATH, PRETRAINED_MODEL = a.pretrained()
    display(json.loads((WORK / 'reports/pretrained_verification.json').read_text()))
else:
    print('Pretrained loading explicitly skipped; fit still verifies the official asset.')

## 9. Model architecture summary
The official pretrained network initially has 1,000 outputs. Standard ClassificationTrainer adapts its classifier to six outputs during fine-tuning. No other architecture modification. Actual trained architecture exports to `fit/six_class_architecture.txt`; until fitting it is NOT AVAILABLE.

In [ ]:
if PRETRAINED_MODEL is not None:
    PRETRAINED_MODEL.info()
    print('Full pretrained architecture:', WORK / 'reports/pretrained_architecture.txt')
    del PRETRAINED_MODEL
    PRETRAINED_MODEL = None
print('Planned training head: six classes; actual output mapping is inspected after fitting.')

## 10. Training configuration
SGD; initial LR 0.01; momentum 0.937; weight decay 0.0005; nominal batch 64 with Ultralytics accumulation/weight-decay scaling; actual minibatch fixed 16. Linear LR schedule to factor 0.01; 3 warmup epochs, initial warmup momentum 0.8 and bias LR 0.1. Actual optimizer groups/LRs exported, including any framework parameter-group details. FP32 (AMP disabled); workers=0 for Windows/notebook reliability; seed42; GPU0 when available, otherwise CPU.

Maximum100 epochs, patience20. Best checkpoint uses internal-validation **top-1 accuracy** (last tied best), via validator fitness override. No hyperparameter optimization. Each run records unique UTC microsecond timestamp + seed + dataset fingerprint. No resume or overwrite. Smoke differs only in 1 epoch, patience1 and a 24-frame technical subset.

In [ ]:
display(pd.Series(CONFIG['training'], name='initial_configuration'))
print('Enabled augmentation: none. Deterministic full-frame letterbox only.')

## 11. Model A training
Full technical QA and actual data-loading verification precede model initialization and fitting. `RUN_TRAINING=True` starts a new official run; defaults only load the latest completed official run for reporting. Each run gets unique paths and source/config/notebook snapshots. Semantic approval is not a gate. Checkpoints save each epoch; an interrupted run's unstripped last.pt retains optimizer state when supported by Ultralytics.

In [ ]:
RUN_DIRECTORY = None
if RUN_TRAINING or RUN_SMOKE_TEST:
    RUN_DIRECTORY = a.run(smoke=RUN_SMOKE_TEST)
    print('Completed run:', RUN_DIRECTORY)
elif (WORK / 'reports/latest_official_run.json').exists():
    RUN_DIRECTORY = Path(json.loads((WORK / 'reports/latest_official_run.json').read_text())['run_directory'])
    print('Displaying existing completed run; no new training:', RUN_DIRECTORY)
else:
    print('No completed official run yet; neither execution flag is enabled.')

## 12. Training curves and early stopping
History comes from this run only. Best epoch reports the last maximum validation top-1, matching tied checkpoint replacement. Early stopping is distinguished from the epoch limit; unavailable framework stop details are labelled rather than invented.

In [ ]:
if RUN_DIRECTORY is not None:
    HISTORY = pd.read_csv(RUN_DIRECTORY / 'training_history.csv')
    curve_columns = [c for c in HISTORY if c in ['train/loss', 'val/loss', 'metrics/accuracy_top1']]
    HISTORY.plot(x='epoch', y=curve_columns, subplots=True, figsize=(9, 7), grid=True)
    plt.tight_layout()
    plt.savefig(RUN_DIRECTORY / 'training_curves.png', dpi=120)
    plt.show()
    EXECUTION = json.loads((RUN_DIRECTORY / 'run_manifest.json').read_text())
    display({k:EXECUTION.get(k) for k in ['mode', 'completed_epochs', 'best_epoch', 'stopping_reason', 'early_stopper_best_epoch']})
else:
    print('Training curves, completed epochs and stopping information: NOT AVAILABLE.')

## 13. Internal validation evaluation
Only the newly trained best checkpoint and internal val split are evaluated. Explicit predictor enforces the same RGB letterbox after reload. Top1, macro precision/recall/F1 and per-class precision/recall/F1/support come from canonical-ID predictions. Confusion rows=true, columns=predicted. Undefined precision/recall/F1 denominators yield0; macro-F1 is the arithmetic mean of **six per-class F1 scores**, not F1 of averaged precision/recall. No detection mAP.

Standalone use: `a.evaluate(best_checkpoint, frozen_rows, new_output_directory, size=224, batch=16, device=0)` or `YOLO(checkpoint).predict(paths, predictor=a.ModelAPredictor, imgsz=224)`. Keep `model_a.py` importable for transform deserialization. Plain default `YOLO.predict` is not the supported inference contract.

In [ ]:
if RUN_DIRECTORY is not None:
    display(json.loads((RUN_DIRECTORY / 'validation_metrics.json').read_text()))
    if RUN_SMOKE_TEST:
        print('TECHNICAL SMOKE ONLY. Official Model A metrics: NOT AVAILABLE.')
else:
    print('Official Model A validation metrics: NOT AVAILABLE. No fit was run.')

## 14. Confusion matrix and per-class errors
Actual output folder names are alphabetical, not research order. `label_mappings.json` records both orders and the output-to-canonical map. Sample exports include IDs, names, canonical labels, model output index, confidence, both full probability vectors and split. Error CSV preserves every mistaken validation sample; no post-hoc label changes.

In [ ]:
if RUN_DIRECTORY is not None:
    display(pd.read_csv(RUN_DIRECTORY / 'per_class_metrics.csv'))
    display(pd.read_csv(RUN_DIRECTORY / 'confusion_matrix.csv', index_col=0))
    display(pd.read_csv(RUN_DIRECTORY / 'validation_errors.csv').head(20))
    from IPython.display import Image as DisplayImage
    display(DisplayImage(filename=str(RUN_DIRECTORY / 'confusion_matrix.png')))
    display(json.loads((RUN_DIRECTORY / 'label_mappings.json').read_text()))
else:
    print('Confusion matrix and error analysis: NOT AVAILABLE.')

## 15. Checkpoint and reproducibility manifest export
Each run retains best.pt and last.pt for later authorized expansion work, together with SHA256 hashes. `run_manifest.json`, `dataset_fingerprint.json`, immutable dataset manifest, explicit exclusions, `actual_training_config.json`, epoch trace, history, architecture and environment bind the run. Do not promote smoke weights to Model A. A failed run is preserved with FAILED status.

In [ ]:
if RUN_DIRECTORY is not None:
    display(json.loads((RUN_DIRECTORY / 'run_manifest.json').read_text()))
else:
    print('Official results: NOT AVAILABLE until the authorized fit completes. No values invented.')

## 16. Actual results and limitations
The full original six-class release has 1,800 selected frames (1,440 train/360 validation), with 240/60 per class and no exclusions. Zero new extraction; all frames come from existing cached candidates. All 480 previous selections retained; 1,320 newly selected with the same declared policy.

Technical quality flags and all 49 historical concern videos remain included and traceable. Historical concerns include ambiguous and invalid decisions and an alternate-label decision on the same selected pilot frames; this exploratory run retains original video labels. Full-frame letterbox preserves available context but cannot repair source clipping, blur or missing temporal information.

Internal validation is not signer-independent; researcher-signers/backgrounds may overlap. Exact hash/pixel checks do not rule out near-duplicates or temporal overlap. No Kaggle development/evaluation, temporal translation, real-time measurement or Model B/C work. Metrics below describe the actual completed internal-validation run only.

In [ ]:
print('Scope: Model A only; internal validation only.')
print('Run:', str(RUN_DIRECTORY) if RUN_DIRECTORY else 'NOT EXECUTED')
print('Results:', 'AVAILABLE for the displayed completed run' if RUN_DIRECTORY else 'NOT AVAILABLE')
print('Quality concerns remain documented; automatic selection does not certify semantics.')